# 01_概述 (Overview)

**来源:** [LangChain Docs — Frontend Overview](https://docs.langchain.com/deep-agents/frontend/overview)

本教程介绍 Deep Agents 前端集成的整体架构和核心模式。Deep Agents 使用协调器-工作者（coordinator-worker）架构，主智能体负责任务规划并委派给专门的子智能体，每个子智能体在隔离环境中运行。前端通过 `useStream` Hook 同时展现协调器的消息和每个子智能体的流式状态。

## 1. 架构概览

| 组件 | 角色 |
|------|------|
| `useStream()` | 前端 Hook，连接后端流式数据 |
| `createDeepAgent()` | 后端 API，创建深度智能体 |
| Subagent A / B | 专门子智能体，隔离运行 |

```
graph LR
  FRONTEND["useStream()"]
  BACKEND["createDeepAgent()"]
  SUB1["Subagent A"]
  SUB2["Subagent B"]

  BACKEND --"stream"--> FRONTEND
  FRONTEND --"submit"--> BACKEND
  BACKEND --"delegate"--> SUB1
  BACKEND --"delegate"--> SUB2
  SUB1 --"result"--> BACKEND
  SUB2 --"result"--> BACKEND
```

## 2. 后端：创建深度智能体

使用 `create_deep_agent` 创建带子智能体的深度智能体后端。

In [ ]:
# 安装依赖
# pip install deepagents langchain-core

# 后端代码示例（运行在 Python 服务端）
from deepagents import create_deep_agent
from dotenv import load_dotenv
load_dotenv()

# 假设有一个 get_weather 工具
# def get_weather(city: str) -> str:
#     return f"{city} 的天气是晴天，25°C"

agent = create_deep_agent(
    model="deepseek-v4-flash",
    tools=[],  # 替换为 [get_weather] 等实际工具
    system_prompt="你是一个乐于助人的智能助手",
    subagents=[
        {
            "name": "researcher",
            "description": "Research assistant",
        }
    ],
)

print(f"深度智能体已创建: {agent}")


## 3. 前端：useStream Hook 连接

在前端，用 `useStream` 连接深度智能体，与普通 `createAgent` 方式相同。深度智能体模式使用额外的 `useStream` 特性：

| 特性 | 用途 |
|------|------|
| `stream.subagents` | 访问所有子智能体及其状态 |
| `stream.values.todos` | 读取智能体的待办事项列表 |
| `filterSubagentMessages` | 分离协调器与子智能体的消息流 |

### React 示例

In [ ]:
# 前端代码（TypeScript/React）：
# 
# import { useStream } from "@langchain/react";
# 
# function App() {
#   const stream = useStream<typeof agent>({
#     apiUrl: "http://localhost:2024",
#     assistantId: "agent",
#   });
# 
#   // 深度智能体特有状态
#   const todos = stream.values?.todos;
#   const subagents = stream.subagents;
# }

print("前端集成要点：useStream 是深度智能体前端集成的核心 Hook")

## 4. 核心模式

| 模式 | 文件 | 说明 |
|------|------|------|
| 子智能体流式 | `03_子Agent流式.ipynb` | 显示专门子智能体的流式内容、进度跟踪和可折叠卡片 |
| 任务列表 | `04_任务列表.ipynb` | 从智能体状态同步实时任务列表，跟踪进度 |
| 沙箱 | `02_沙箱.ipynb` | 构建 IDE 风格的 UI，包含文件浏览器、代码查看器和差异面板 |

## 5. 相关资源

LangChain 的前端模式（Markdown 消息、工具调用、人在回路等）同样适用于深度智能体。Deep Agents 基于相同的 LangGraph 运行时构建，因此 `useStream` 提供相同的核心 API。

---

**参考链接**
- [LangChain Docs — Frontend Overview](https://docs.langchain.com/deep-agents/frontend/overview)
- [@langchain/react 文档](https://docs.langchain.com/react)
- [useStream API 参考](https://docs.langchain.com/react/use-stream)
- [Deep Agents GitHub](https://github.com/langchain-ai/deep-agents)